In [143]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

from utils import *

In [144]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [145]:
stoi,itos = stoi_and_itos(words)

In [153]:
block_size = 3
(Xtr, Ytr), (Xdev, Ydev), (Xte, Yte) = split_dataset(words, block_size, [0.8, 0.1, 0.1], stoi)


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [154]:
C = torch.randn((27,2))
#C[Xtr]

In [166]:
emb_dim = 10                  
n_hidden = 200                  
vocab_size = len(stoi)          
input_size = block_size * emb_dim

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((vocab_size, emb_dim), generator=g)
W1 = torch.randn((input_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden, generator=g)
W2 = torch.randn((n_hidden, vocab_size), generator=g)
b2 = torch.randn(vocab_size, generator=g)
parameters = [C, W1, b1, W2, b2]

print(sum(p.nelement() for p in parameters))

for p in parameters:
  p.requires_grad = True


11897


In [167]:
emb = C[Xtr]
input_lay = emb.view(-1, input_size)
hidden = torch.tanh(input_lay @ W1 + b1)
logits = hidden @ W2 + b2

# logits = logits - max(logits)
# counts = logits.exp() 
# prob = counts / counts.sum(1,keepdims=True)
# loss = -prob[torch.arange(Ytr.shape[0]), Ytr].log().mean()
loss = F.cross_entropy(logits,Ytr) #same as the code above

loss.item()
# forward pass of the neural net

26.004695892333984

In [ ]:

# same but bathced


batch_size = 32
for i in range(2000):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,))

    # emb = C[Xtr]
    emb = C[Xtr[ix]]
   
    input_lay = emb.view(-1, input_size)
    hidden = torch.tanh(input_lay @ W1 + b1)
    logits = hidden @ W2 + b2

    #loss = F.cross_entropy(logits,Ytr)
    loss = F.cross_entropy(logits,Ytr[ix])


    print(loss.item())

    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad



2.4661755561828613
4.317375659942627
3.8603742122650146
3.4424192905426025
3.2377991676330566
2.716670036315918
2.4967026710510254
3.066828966140747
2.6350796222686768
2.519484043121338
3.100123882293701
2.792407512664795
2.9703073501586914
2.9496700763702393
3.1523022651672363
2.6962685585021973
2.7655558586120605
3.5855495929718018
2.928995370864868
2.6729536056518555
2.9768176078796387
2.9537417888641357
3.2873008251190186
2.918233871459961
2.7904863357543945
3.534176826477051
3.7700693607330322
3.669605255126953
2.7789502143859863
2.65494966506958
2.9476664066314697
2.919574022293091
2.940481185913086
2.7821598052978516
3.1324243545532227
2.671315908432007
2.8193938732147217
2.670534133911133
3.1429102420806885
2.651045083999634
2.7492549419403076
2.816239356994629
2.5868418216705322
2.7801098823547363
2.991971015930176
2.923093795776367
3.126831531524658
3.0850255489349365
2.8006749153137207
3.0253589153289795
2.741029977798462
2.7681539058685303
2.5447874069213867
3.0018603801727

In [165]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

cartahxatille.
lhi.
mlix.
tatyl.
nla.
taenl.
hnen.
amelyht.
taqei.
neltanttceniiy.
asletyyhh.
mon.
anteniyn.
anlel.
alia.
biyi.
jelo.
dearyxi.
jajenli.
sat.
